
## Theory

Logistic Regression is a supervised machine learning algorithm used for classification problems, especially binary classification. It predicts the probability that an observation belongs to a particular class.

First, a linear combination of input features is calculated:

z = b + w1x1 + w2x2 + ... + wnxn

The sigmoid function converts this value into a probability between 0 and 1:

P(y=1) = 1 / (1 + e^(-z))

A threshold of 0.5 is used to classify the output. If the predicted probability is greater than or equal to 0.5, the output is classified as class 1; otherwise, it is classified as class 0.

The model learns its weights and bias using gradient descent, which minimizes binary cross-entropy loss. Its performance can be evaluated using accuracy, precision, recall, and F1-score.

In this practical, logistic regression is implemented from scratch using mathematical formulas to predict whether a student passes (1) or fails (0) based on five normalized subject marks.


In [1]:

import pandas as pd
import math
import random

# Load the processed dataset
df = pd.read_excel("EXPORTS/Mldata_Preprocessed.xlsx")

# Clean column names
df.columns = df.columns.str.strip()

print("Dataset loaded successfully!")
print("Dataset shape:", df.shape)
print("\nColumns:", df.columns.tolist())

df.head()


Dataset loaded successfully!
Dataset shape: (67, 9)

Columns: ['ROLL NO.', 'GENDER', 'SOFTWARE ENGINEER', 'TOC', 'COMPUTER NETWORK', 'MACHINE LEARNING', 'DATA WAREHOUSE', 'GRADE', 'RESULT']


,ROLL NO.,GENDER,SOFTWARE ENGINEER,TOC,COMPUTER NETWORK,MACHINE LEARNING,DATA WAREHOUSE,GRADE,RESULT
0,240104010001,M,0.500000,0.491525,0.482759,0.473684,0.454545,C,1
1,240104010002,F,0.500000,0.661017,0.741379,0.649123,0.545455,B,1
2,240104010003,M,0.633333,0.627119,0.620690,0.614035,0.600000,B,1
3,240104010004,M,0.983333,0.915254,0.948276,0.964912,0.890909,O,1
4,240104010005,M,0.500000,0.491525,0.482759,0.473684,0.454545,C,1


In [2]:

# Five input features
features = [
    "SOFTWARE ENGINEER",
    "TOC",
    "COMPUTER NETWORK",
    "MACHINE LEARNING",
    "DATA WAREHOUSE"
]

# Binary target variable
target = "RESULT"

# Extract data as standard Python lists
X = [
    [float(row[feature]) for feature in features]
    for _, row in df.iterrows()
]

y = [int(value) for value in df[target].tolist()]

print("Input features:", features)
print("Target variable:", target)
print("Total records:", len(X))
print("Pass records:", sum(y))
print("Fail records:", len(y) - sum(y))


Input features: ['SOFTWARE ENGINEER', 'TOC', 'COMPUTER NETWORK', 'MACHINE LEARNING', 'DATA WAREHOUSE']
Target variable: RESULT
Total records: 67
Pass records: 64
Fail records: 3


In [3]:

# Separate record indices by class
class_0 = [i for i in range(len(y)) if y[i] == 0]
class_1 = [i for i in range(len(y)) if y[i] == 1]

rng = random.Random(42)
rng.shuffle(class_0)
rng.shuffle(class_1)

def split_class(indices):
    if len(indices) <= 1:
        return indices[:], []

    test_count = max(1, round(len(indices) * 0.20))
    test_count = min(test_count, len(indices) - 1)

    return indices[test_count:], indices[:test_count]

train_0, test_0 = split_class(class_0)
train_1, test_1 = split_class(class_1)

train_indices = train_0 + train_1
test_indices = test_0 + test_1

rng.shuffle(train_indices)
rng.shuffle(test_indices)

X_train = [X[i] for i in train_indices]
y_train = [y[i] for i in train_indices]

X_test = [X[i] for i in test_indices]
y_test = [y[i] for i in test_indices]

print("Training records:", len(X_train))
print("Testing records:", len(X_test))
print("Pass records in training:", sum(y_train))
print("Fail records in training:", len(y_train) - sum(y_train))
print("Pass records in testing:", sum(y_test))
print("Fail records in testing:", len(y_test) - sum(y_test))


Training records: 53
Testing records: 14
Pass records in training: 51
Fail records in training: 2
Pass records in testing: 13
Fail records in testing: 1


In [4]:

def sigmoid(z):
    # Prevent overflow for very large positive or negative values
    z = max(-500.0, min(500.0, z))
    return 1.0 / (1.0 + math.exp(-z))


def predict_probability(weights, bias, row):
    z = bias + sum(
        weights[j] * row[j]
        for j in range(len(weights))
    )

    return sigmoid(z)


print("Sigmoid(0):", sigmoid(0))
print("Sigmoid(2):", sigmoid(2))
print("Sigmoid(-2):", sigmoid(-2))


Sigmoid(0): 0.5
Sigmoid(2): 0.8807970779778823
Sigmoid(-2): 0.11920292202211755


In [5]:

def train_logistic_regression(
    X_train, y_train, learning_rate=0.1, epochs=5000
):
    n_samples = len(X_train)
    n_features = len(X_train[0])

    # Initialize weights and bias
    weights = [0.0] * n_features
    bias = 0.0

    for epoch in range(epochs):
        weight_gradients = [0.0] * n_features
        bias_gradient = 0.0

        # Calculate gradients over the training data
        for i in range(n_samples):
            probability = predict_probability(
                weights, bias, X_train[i]
            )

            error = probability - y_train[i]

            for j in range(n_features):
                weight_gradients[j] += error * X_train[i][j]

            bias_gradient += error

        # Average the gradients and update parameters
        for j in range(n_features):
            weights[j] -= (
                learning_rate * weight_gradients[j] / n_samples
            )

        bias -= learning_rate * bias_gradient / n_samples

    return weights, bias


weights, bias = train_logistic_regression(
    X_train,
    y_train,
    learning_rate=0.1,
    epochs=5000
)

print("Training completed!")
print("\nLearned weights:")

for feature, weight in zip(features, weights):
    print(f"{feature}: {weight:.6f}")

print("\nBias:", round(bias, 6))


Training completed!

Learned weights:
SOFTWARE ENGINEER: 2.675289
TOC: 1.520441
COMPUTER NETWORK: 4.676443
MACHINE LEARNING: 1.901709
DATA WAREHOUSE: -0.578945

Bias: -1.820228


In [6]:

def evaluate_model(X_test, y_test, weights, bias):
    probabilities = [
        predict_probability(weights, bias, row)
        for row in X_test
    ]

    predictions = [
        1 if p >= 0.5 else 0
        for p in probabilities
    ]

    tp = sum(1 for a, p in zip(y_test, predictions) if a == 1 and p == 1)
    tn = sum(1 for a, p in zip(y_test, predictions) if a == 0 and p == 0)
    fp = sum(1 for a, p in zip(y_test, predictions) if a == 0 and p == 1)
    fn = sum(1 for a, p in zip(y_test, predictions) if a == 1 and p == 0)

    accuracy = (tp + tn) / len(y_test)
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1 = (
        2 * precision * recall / (precision + recall)
        if precision + recall else 0.0
    )

    epsilon = 1e-15
    loss = -sum(
        actual * math.log(max(epsilon, p))
        + (1 - actual) * math.log(max(epsilon, 1 - p))
        for actual, p in zip(y_test, probabilities)
    ) / len(y_test)

    print("Confusion Matrix:")
    print("                 Predicted Pass  Predicted Fail")
    print(f"Actual Pass          {tp:5d}           {fn:5d}")
    print(f"Actual Fail          {fp:5d}           {tn:5d}")

    print("\nEvaluation Metrics:")
    print(f"Accuracy:  {accuracy * 100:.2f}%")
    print(f"Precision: {precision:.4f}")
    print(f"Recall:    {recall:.4f}")
    print(f"F1-score:  {f1:.4f}")
    print(f"Log Loss:  {loss:.4f}")


evaluate_model(X_test, y_test, weights, bias)


Confusion Matrix:
                 Predicted Pass  Predicted Fail
Actual Pass             13               0
Actual Fail              1               0

Evaluation Metrics:
Accuracy:  92.86%
Precision: 0.9286
Recall:    1.0000
F1-score:  0.9630
Log Loss:  0.1035


In [7]:

# Minimum and maximum values from the original dataset
feature_min = {
    "SOFTWARE ENGINEER": 0,
    "TOC": 1,
    "COMPUTER NETWORK": 2,
    "MACHINE LEARNING": 3,
    "DATA WAREHOUSE": 5
}

feature_max = {
    "SOFTWARE ENGINEER": 60,
    "TOC": 60,
    "COMPUTER NETWORK": 60,
    "MACHINE LEARNING": 60,
    "DATA WAREHOUSE": 60
}


def minmax_normalize(subject, marks):
    minimum = feature_min[subject]
    maximum = feature_max[subject]

    return (marks - minimum) / (maximum - minimum)


raw_marks = {}

print("Enter the student's original subject marks:\n")

for subject in features:
    while True:
        try:
            marks = float(input(f"{subject} marks: "))

            if not feature_min[subject] <= marks <= feature_max[subject]:
                print(
                    f"Please enter a value between "
                    f"{feature_min[subject]} and {feature_max[subject]}."
                )
                continue

            raw_marks[subject] = marks
            break

        except ValueError:
            print("Invalid input. Please enter a number.")

# Normalize the input using the original dataset's ranges
normalized_input = [
    minmax_normalize(subject, raw_marks[subject])
    for subject in features
]

print("\nNormalized Input Features:")

for subject, value in zip(features, normalized_input):
    print(f"{subject}: {value:.6f}")


Enter the student's original subject marks:

Please enter a value between 5 and 60.

Normalized Input Features:
SOFTWARE ENGINEER: 0.750000
TOC: 0.559322
COMPUTER NETWORK: 0.982759
MACHINE LEARNING: 0.842105
DATA WAREHOUSE: 0.000000


In [8]:

# Calculate the probability of passing
pass_probability = predict_probability(
    weights, bias, normalized_input
)

# Apply the classification threshold
predicted_result = 1 if pass_probability >= 0.5 else 0

print("\n--- Logistic Regression Prediction ---")

print(f"Probability of passing: {pass_probability * 100:.2f}%")
print(f"Probability of failing: {(1 - pass_probability) * 100:.2f}%")

if predicted_result == 1:
    print("Predicted RESULT: 1 (PASS)")
else:
    print("Predicted RESULT: 0 (FAIL)")



--- Logistic Regression Prediction ---
Probability of passing: 99.93%
Probability of failing: 0.07%
Predicted RESULT: 1 (PASS)
